# 05 - Score Answer Quality Carefully

Evaluates generated answer quality. The default path is offline and uses an extractive context answer plus lexical relevance. Optional DeepEval scoring is available when API access is desired.


## Learning Goal

Move from evidence retrieval to answer quality without pretending that a vague quality score is enough. This lab starts with lightweight offline proxies and keeps LLM-as-judge scoring optional so students can see the cost and calibration tradeoff.

## Where This Fits

Progression: data sanity -> router eval -> retrieval eval -> cascade eval -> answer quality -> full benchmark -> ablation.

This is the point where evaluation becomes more subjective. The main lesson is to be cautious: answer-quality evals need clear criteria, human labels, or careful judge validation.

## Related AI Evals Concepts

- How To Trust A LLM Judge: judge scores need calibration against human labels before they become decision metrics.
- Don't Use Likert Scales: prefer targeted pass/fail or specific sub-metrics over broad 1-5 quality ratings.
- Types Of Automated Evals: combine cheap proxy checks with optional LLM-as-judge scoring.
- AI Eval Mistakes: do not outsource judgment to a model before understanding the data and failure modes.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT


PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag')

In [2]:
import importlib
from typing import Any

import chromadb
import pandas as pd

import agentic_rag.ingestion as ingestion_module  # noqa: E402
import agentic_rag.retrievers as retrievers_module  # noqa: E402
import agentic_rag.router as router_module  # noqa: E402
from agentic_rag.constants import SourceType  # noqa: E402

importlib.reload(ingestion_module)
importlib.reload(retrievers_module)
importlib.reload(router_module)

from agentic_rag.ingestion import ensure_chroma_collections  # noqa: E402
from agentic_rag.retrievers import ChromaRetriever  # noqa: E402
from agentic_rag.settings import Settings  # noqa: E402
from agentic_rag.telemetry import configure_tracing, start_span  # noqa: E402

pd.set_option("display.max_colwidth", 180)


In [3]:
TRACE_DIR = PROJECT_ROOT / "otel_traces"
TRACE_DIR.mkdir(parents=True, exist_ok=True)
TRACE_FILE = TRACE_DIR / "05_answer_quality_deepeval.jsonl"

trace_settings = Settings(
    _env_file=None,
    OTEL_TRACING_ENABLED=True,
    OTEL_TRACES_EXPORTER="file",
    OTEL_TRACES_FILE=TRACE_FILE,
    OTEL_SERVICE_NAME="agentic-rag-notebooks",
)
configure_tracing(trace_settings)

TRACE_FILE


PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/otel_traces/05_answer_quality_deepeval.jsonl')

## Load Answer-Quality Test Sets


In [4]:
base_df = pd.read_csv(PROJECT_ROOT / "datasets/evaluation_dataset.csv")
challenge_df = pd.read_csv(PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv")

if "Query" in base_df.columns:
    base_df = base_df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})

base_df["dataset"] = "base"
challenge_df["dataset"] = "challenging"
base_df["expected_source_type"] = base_df["expected_source_type"].astype(str)
challenge_df["expected_source_type"] = challenge_df["expected_source_type"].astype(str)

datasets = pd.concat([base_df, challenge_df], ignore_index=True, sort=False)
datasets[["dataset", "query", "expected_source_type"]].head()


,dataset,query,expected_source_type
0,base,How many people are affected by X-linked chondrodysplasia punctata 1 ?,Retrieve_QnA
1,base,What are the treatments for Kawasaki disease ?,Retrieve_QnA
2,base,What are the genetic changes related to Ellis-van Creveld syndrome ?,Retrieve_QnA
3,base,What are the symptoms of Renal dysplasia-limb defects syndrome ?,Retrieve_QnA
4,base,What is (are) Fraser syndrome ?,Retrieve_QnA


In [5]:
qna_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_qna_dataset.csv")
device_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_device_manuals_dataset.csv")

settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")
client = chromadb.PersistentClient(path=str(settings.chroma_path))

collection_summary = ensure_chroma_collections(client, qna_df, device_df)
collection_summary


,collection,exists,count
0,medical_qna,True,16407
1,medical_device_manual,True,2694


## Define Answer-Quality Proxies And Optional Judge


In [11]:
LABELS = [source.value for source in SourceType]
LOCAL_SOURCES = {SourceType.RETRIEVE_QNA.value, SourceType.RETRIEVE_DEVICE.value}
settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")


def source_or_none(value: str) -> SourceType | None:
    try:
        return SourceType(value)
    except ValueError:
        return None


def is_local_source(value: str) -> bool:
    return value in LOCAL_SOURCES


def lexical_score(question: str, answer: str) -> float:
    q_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in question.split() if len(token) > 3}
    a_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in answer.split() if len(token) > 3}
    if not q_tokens or not a_tokens:
        return 0.0
    return len(q_tokens & a_tokens) / len(q_tokens)


def preview_text(text: str, max_words: int = 50) -> str:
    return " ".join(text.split()[:max_words])

RUN_DEEPEVAL = True

async def answer_one(row: pd.Series, top_k: int = 3) -> dict[str, Any]:
    query = str(row["query"])
    expected_source = str(row["expected_source_type"])
    expected_answer = str(row.get("expected_answer") or "")

    with start_span("notebook.answer_quality.row", dataset=str(row["dataset"]), query_length=len(query)):
        context_docs: list[str] = []
        if is_local_source(expected_source):
            retriever = ChromaRetriever(chroma_path=str(settings.chroma_path), top_k=top_k)
            docs = await retriever.retrieve(SourceType(expected_source), query)
            context_docs = [doc.text for doc in docs]

        actual_answer = preview_text(context_docs[0], max_words=50) if context_docs else ""
        lexical_question_relevance = lexical_score(query, actual_answer) if actual_answer else None
        lexical_expected_overlap = lexical_score(expected_answer, actual_answer) if expected_answer and actual_answer else None

        deepeval_score = None
        deepeval_reason = ""
        deepeval_success = None
        if RUN_DEEPEVAL and actual_answer:
            from deepeval.metrics import AnswerRelevancyMetric
            from deepeval.test_case import LLMTestCase

            metric = AnswerRelevancyMetric(threshold=0.7, model=settings.evaluator_model, include_reason=True, async_mode=False)
            test_case = LLMTestCase(
                input=query,
                actual_output=actual_answer,
                expected_output=expected_answer or None,
                retrieval_context=context_docs or None,
            )
            metric.measure(test_case)
            deepeval_score = metric.score
            deepeval_reason = metric.reason
            deepeval_success = metric.success

        return {
            "dataset": row["dataset"],
            "query": query,
            "expected_source_type": expected_source,
            "actual_answer": actual_answer,
            "expected_answer": expected_answer,
            "context_count": len(context_docs),
            "lexical_question_relevance": lexical_question_relevance,
            "lexical_expected_overlap": lexical_expected_overlap,
            "answer_relevancy": deepeval_score,
            "answer_relevancy_reason": deepeval_reason,
            "answer_relevancy_success": deepeval_success,
            "metric_status": "deepeval_scored" if deepeval_score is not None else ("offline_proxy" if actual_answer else "skipped_no_local_context"),
        }


async def evaluate_answers(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        rows.append(await answer_one(row))
    return pd.DataFrame(rows)


## Start With Offline Answer-Quality Proxies


In [12]:
base_answer_results = await evaluate_answers(base_df)
base_answer_results.head()


/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/.venv/lib/python3.14/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/.venv/lib/python3.14/site-packages/deepeval/evaluate/execute/loop.py:809: SyntaxWarning: 'return' in a 'finally' block
  return


/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/.venv/lib/python3.14/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/.venv/lib/python3.14/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/.venv/lib/python3.14/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/.venv/lib/python3.14/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/.venv/lib/python3.14/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/.venv/lib/python3.14/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/.venv/lib/python3.14/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

,dataset,query,expected_source_type,actual_answer,expected_answer,context_count,lexical_question_relevance,lexical_expected_overlap,answer_relevancy,answer_relevancy_reason,answer_relevancy_success,metric_status
0,base,How many people are affected by X-linked chondrodysplasia punctata 1 ?,Retrieve_QnA,Question: How many people are affected by X-linked chondrodysplasia punctata 1 ? Answer: The prevalence of X-linked chondrodysplasia punctata 1 is unknown. Several dozen affect...,The prevalence of X-linked chondrodysplasia punctata 1 is unknown. Several dozen affected males have been reported in the scientific literature.,3,1.000000,1.000000,1.0,"The score is 1.00 because there were no irrelevant statements in the actual output (the list is empty), so irrelevance cannot lower the score; thus it cannot be higher due to i...",True,deepeval_scored
1,base,What are the treatments for Kawasaki disease ?,Retrieve_QnA,Question: What are the treatments for Tubular aggregate myopathy ? Answer: How might tubular aggregate myopathy be treated?,These resources address the diagnosis or management of Kawasaki disease: - Cincinnati Children's Hospital Medical Center - Genetic Testing Registry: Acute febrile mucocutaneo...,3,0.500000,0.022727,0.0,The score is 0.00 because the actual output contained off-topic statements about Tubular aggregate myopathy instead of addressing Kawasaki disease treatments; these irrelevant ...,False,deepeval_scored
2,base,What are the genetic changes related to Ellis-van Creveld syndrome ?,Retrieve_QnA,Question: What are the genetic changes related to Ellis-van Creveld syndrome ? Answer: Ellis-van Creveld syndrome can be caused by mutations in the EVC or EVC2 gene. Little is ...,"Ellis-van Creveld syndrome can be caused by mutations in the EVC or EVC2 gene. Little is known about the function of these genes, although they appear to play important roles i...",3,1.000000,0.307692,0.5,The score is 0.50 because the output focused on general gene function and development rather than listing or describing the specific mutations causing Ellis-van Creveld syndrom...,False,deepeval_scored
3,base,What are the symptoms of Renal dysplasia-limb defects syndrome ?,Retrieve_QnA,Question: What are the symptoms of Bell's palsy ? Answer: What are the symptoms of Bell's palsy?,What are the signs and symptoms of Renal dysplasia-limb defects syndrome? The Human Phenotype Ontology provides the following list of signs and symptoms for Renal dysplasia-lim...,3,0.333333,0.012987,1.0,"The score is 1.00 because there are no irrelevant statements in the output; it stays on-topic, and addressing renal dysplasia-limb defects syndrome symptoms would be the next s...",True,deepeval_scored
4,base,What is (are) Fraser syndrome ?,Retrieve_QnA,"Question: What is (are) Cri du chat syndrome ? Answer: Cri du chat syndrome, also known as 5p- (5p minus) syndrome or cat cry syndrome, is a genetic condition that is caused by...",Fraser syndrome is a rare disorder that affects development starting before birth. Characteristic features of this condition include eyes that are completely covered by skin an...,3,0.750000,0.043478,0.0,"The score is 0.00 because all content was off-topic, discussing Cri du chat syndrome instead of Fraser syndrome. It cannot be higher since none of the output addressed Fraser s...",False,deepeval_scored


In [13]:
base_answer_summary = base_answer_results.groupby(["dataset", "expected_source_type", "metric_status"], dropna=False).agg(
    examples=("query", "count"),
    lexical_question_relevance=("lexical_question_relevance", "mean"),
    lexical_expected_overlap=("lexical_expected_overlap", "mean"),
    answer_relevancy=("answer_relevancy", "mean"),
).reset_index()
base_answer_summary


,dataset,expected_source_type,metric_status,examples,lexical_question_relevance,lexical_expected_overlap,answer_relevancy
0,base,Retrieve_Device,deepeval_scored,12,0.384888,0.222358,0.240079
1,base,Retrieve_QnA,deepeval_scored,12,0.731944,0.248679,0.562500
2,base,Web_Search,skipped_no_local_context,3,NaN,NaN,NaN


## Inspect Hard Answer-Quality Cases


In [9]:
challenge_answer_results = await evaluate_answers(challenge_df)
challenge_answer_results.head()


,dataset,query,expected_source_type,actual_answer,expected_answer,context_count,lexical_question_relevance,lexical_expected_overlap,answer_relevancy,answer_relevancy_reason,answer_relevancy_success,metric_status
0,challenging,The manual says the Model 1606 Electrosurgical Unit is contraindicated near MRI environments; what general risks do MRI environments create for implanted devices?,Retrieve_Device,Device_Name: Ultrasound Scanner Model_Number: Plus388 Manufacturer: Roche Patient_Population: Adult (>65) Indications_for_Use: Intended for surgical neurological procedures in ...,,3,0.000000,None,None,,None,offline_proxy
1,challenging,"For a child with fever and rash after vaccination, should I use the Q&A knowledge base or search current outbreak updates?",Web_Search,,,0,NaN,None,None,,None,skipped_no_local_context
2,challenging,What are the symptoms of Kawasaki disease and are there any new FDA-approved devices used to monitor it this year?,Web_Search,,,0,NaN,None,None,,None,skipped_no_local_context
3,challenging,"Does the Pro127 Electrosurgical Unit support pediatric use, and what symptoms would indicate a complication?",Retrieve_Device,Device_Name: Electrosurgical Unit Model_Number: BEC965 Manufacturer: Beckman Coulter Patient_Population: All Indications_for_Use: Used for emergency hemodialysis in acute strok...,,3,0.166667,None,None,,None,offline_proxy
4,challenging,What changed recently in contraindications for dialysis machines from major manufacturers?,Web_Search,,,0,NaN,None,None,,None,skipped_no_local_context


In [10]:
challenge_answer_results[["query", "expected_source_type", "actual_answer", "lexical_question_relevance", "metric_status"]]


,query,expected_source_type,actual_answer,lexical_question_relevance,metric_status
0,The manual says the Model 1606 Electrosurgical Unit is contraindicated near MRI environments; what general risks do MRI environments create for implanted devices?,Retrieve_Device,Device_Name: Ultrasound Scanner Model_Number: Plus388 Manufacturer: Roche Patient_Population: Adult (>65) Indications_for_Use: Intended for surgical neurological procedures in ...,0.000000,offline_proxy
1,"For a child with fever and rash after vaccination, should I use the Q&A knowledge base or search current outbreak updates?",Web_Search,,NaN,skipped_no_local_context
2,What are the symptoms of Kawasaki disease and are there any new FDA-approved devices used to monitor it this year?,Web_Search,,NaN,skipped_no_local_context
3,"Does the Pro127 Electrosurgical Unit support pediatric use, and what symptoms would indicate a complication?",Retrieve_Device,Device_Name: Electrosurgical Unit Model_Number: BEC965 Manufacturer: Beckman Coulter Patient_Population: All Indications_for_Use: Used for emergency hemodialysis in acute strok...,0.166667,offline_proxy
4,What changed recently in contraindications for dialysis machines from major manufacturers?,Web_Search,,NaN,skipped_no_local_context
5,Compare the contraindications of the Max787 Dialysis Machine with standard contraindications for dialysis patients.,Retrieve_Device,Device_Name: Surgical Drill Model_Number: Pro721 Manufacturer: Synthes Patient_Population: Pediatric (2-18) Indications_for_Use: Intended for continuous monitoring evaluation i...,0.375000,offline_proxy
6,What is Fraser syndrome and can any surgical robot in our manuals be used for related procedures?,Retrieve_Device,Device_Name: Surgical Robot Model_Number: Max578 Manufacturer: Boston Scientific Patient_Population: Adult and Pediatric Indications_for_Use: Used for post-operative chemothera...,0.333333,offline_proxy
7,Which device should be used for emergency pulmonary stabilization in pediatric patients?,Retrieve_Device,Device_Name: Catheter Model_Number: Pro753 Manufacturer: Karl Storz Patient_Population: Adult and Pediatric Indications_for_Use: Used for long-term fluid resuscitation delivery...,0.333333,offline_proxy
8,What are the latest recommendations for treating Pompe disease?,Web_Search,,NaN,skipped_no_local_context
9,"If the retrieved Q&A answer about dystonia is irrelevant, should the graph fall back to Tavily?",Web_Search,,NaN,skipped_no_local_context


## Pay Attention To

- Answer quality is harder to score than routing or retrieval because it can be subjective.
- Lightweight lexical proxies are useful for iteration, but they are not a substitute for human judgment.
- LLM-as-judge evaluation should be optional until you can compare it against trusted labels.
- Keep the question narrow: score a specific failure mode rather than overall helpfulness.


## Optional Advanced Path: DeepEval LLM Judge

The `RUN_DEEPEVAL` flag in the helper cell enables model-graded answer relevancy. Treat this as an advanced path: it adds cost and latency, and its scores should be checked against human labels before being used as a source of truth.


## Export Answer-Quality Artifacts


In [ ]:
answer_results = pd.concat([base_answer_results, challenge_answer_results], ignore_index=True)
answer_summary = pd.concat([base_answer_summary], ignore_index=True)

results_path = PROJECT_ROOT / "output/answer_quality_results.csv"
summary_path = PROJECT_ROOT / "output/answer_quality_summary.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.parent.mkdir(parents=True, exist_ok=True)
answer_results.to_csv(results_path, index=False)
answer_summary.to_csv(summary_path, index=False)

results_path, summary_path
